# Results viewer — her vs my models

Everything the pipeline produced, as **plots and numbers tables** (no JSON reading needed).
Run `run_pipeline.py` first, set `RUN` below to the profile you ran (`full` or `quick`), then **Run All**.

**Score** = the CW3 marking metric: per-class recall, averaged.
- HAR = mean of (static mean, dynamic mean).
- Social = balanced recall over the 4 classes.

All numbers are % and mean ± std over runs (folds × seeds).

| scenario | train on → test on | folds? |
|---|---|---|
| **A** | your real data → your real data | yes (within-subject CV) |
| **B** | all real → all Jan2026 | no (different datasets; seeds give the spread) |
| **C** | all Jan2026 → all real | no |
| **D** | Jan2026 → Jan2026 | yes (Lab3 B9 10-fold) |
| **E** | Jan2026 pre-train + real fine-tune → real | yes (same folds as A) |

Groups compared: **MLP, CNN, RNN, FEATURE**. Mine are blue, solid and unhatched; hers are orange (or other
colours), dashed or hatched.

In [ ]:
# ── settings: change these and re-run the cells below ─────────────────────────
RUN = "full"            # which runs/<RUN>/ folder to read ("full" or "quick")
TASK = "social"         # "har" or "social" — used by the detail sections (6+)
SCENARIO = "D"          # "A".."E"       — used by the detail sections
GROUP = "CNN"           # "MLP" | "CNN" | "RNN" | "FEATURE"
MODEL = "harry_cnn"     # for the per-epoch table and confusion matrix

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import display, Markdown
%matplotlib inline

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from harry import tables as T, viz
from harry.zoo import ZOO, GROUPS
from harry.tables import VIEWS, VIEW_TITLES

OUT = ROOT / "runs" / RUN
assert (OUT / "jobs").exists(), f"{OUT} has no results yet — run run_pipeline.py (profile {RUN}) first"
tb = T.write(OUT)                     # (re)build tables/*.csv from the finished jobs — always up to date
runs, pcls, eps, conf, dep, status = (tb[k] for k in ("runs", "per_class", "epochs", "confusion", "deploy", "status"))
pd.set_option("display.max_columns", 60, "display.width", 250)
pct = lambda df: df.style.format(lambda v: f"{100*v:.1f}" if isinstance(v, (float, np.floating)) and v == v else v)
display(Markdown(f"**{len(runs)}** test results from **{runs.groupby(['task','scenario','model','fold','seed']).ngroups if len(runs) else 0}** trained models · tables in `{OUT/'tables'}`"))

## 1 · Progress
How many jobs are finished, failed or pending per scenario. You can open this notebook while the pipeline is still running.

In [ ]:
viz.progress(status)

## 2 · Leaderboard — every model × every scenario
Darker = better. Bold = best in that scenario. The table below has the same numbers.

In [ ]:
for task in [t for t in ["har", "social"] if t in set(runs.task)]:
    display(Markdown(f"### {task.upper()}"))
    display(viz.leaderboard_heatmap(runs, task))
    lb, sd = T.leaderboard(runs, task)
    display(lb.style.format(lambda v: f"{100*v:.1f}" if isinstance(v, float) and v == v else ("–" if isinstance(v, float) else v))
              .background_gradient(cmap=viz.BLUES, vmin=0.25, vmax=1.0, subset=[c for c in VIEWS if c in lb.columns]))

## 3 · Best model for every circumstance
The winner of each scenario, and in each group the best of hers → the best of mine (difference in percentage points).

In [ ]:
for task in [t for t in ["har", "social"] if t in set(runs.task)]:
    display(Markdown(f"### {task.upper()}"))
    display(T.best_per_circumstance(runs, task))
    display(viz.her_vs_mine(runs, task))

## 4 · CW3 targets
- **HAR:** static mean ≥ 90 %, dynamic mean ≥ 85 %.
- **Social:** every class ≥ 75 %.

✗ marks a miss.

In [ ]:
for task in [t for t in ["har", "social"] if t in set(runs.task)]:
    for v in [v for v in VIEWS if v in set(runs[runs.task == task].scenario)]:
        display(viz.requirements(runs, pcls, task, v))

## 5 · Group by group (MLP → CNN → RNN → FEATURE)
For each group: the score per scenario, then for each scenario a numbers table and the learning curves.

The curves show training loss, training accuracy, and the held-out score after every epoch. Training uses a fixed number of epochs; the held-out curve is only for viewing.

In [ ]:
for grp in GROUPS:
    models = [m for m in T.model_order() if ZOO[m].group == grp]
    for task in [t for t in ["har", "social"] if t in set(runs.task)]:
        if runs[(runs.task == task) & (runs.group == grp)].empty:
            continue
        display(Markdown(f"### {grp} · {task.upper()} — " + ", ".join(models)))
        display(viz.group_scores(runs, task, grp))
        for v in VIEWS:
            gt = T.group_table(runs, pcls, task, v, models)
            if gt.empty:
                continue
            display(Markdown(f"**{VIEW_TITLES[v]}**"))
            display(gt)
            display(viz.learning_curves(eps, task, v, models))

## 6 · Per-epoch numbers for one model
Uses `TASK`, `SCENARIO` and `MODEL` from the settings cell. Values are the mean over runs at each epoch; the second table is the last epoch of every individual run.

In [ ]:
e = eps[(eps.task == TASK) & (eps.scenario == SCENARIO) & (eps.model == MODEL)]
if e.empty:
    print("no history for", TASK, SCENARIO, MODEL)
else:
    cols = ["train_loss", "train_acc", "heldout_loss", "heldout_acc", "heldout_score", "epoch_seconds"]
    display(e.groupby("epoch")[cols].mean().round(4))
    display(e.sort_values("epoch").groupby(["fold", "seed"]).tail(1)[["fold", "seed", "epoch"] + cols].round(4))

## 7 · Recall of every class
Uses `TASK` and `SCENARIO`. This shows which classes are confused.

In [ ]:
display(viz.per_class_heatmap(pcls, TASK, SCENARIO))
pc = pcls[(pcls.task == TASK) & (pcls.scenario == SCENARIO)]
display(pc.pivot_table(index="model", columns="class", values="recall", aggfunc="mean").reindex(columns=T.classes_of(TASK)).mul(100).round(1))

## 8 · Spread over folds and seeds
Each dot is one trained model. A difference between two models smaller than these boxes is noise.

In [ ]:
for v in [v for v in VIEWS if v in set(runs[runs.task == TASK].scenario)]:
    display(viz.fold_spread(runs, TASK, v))

## 9 · Confusion matrices
Uses `TASK`, `SCENARIO` and `GROUP`. Each row is a true class, normalised to 100 %.

In [ ]:
for m in [m for m in T.model_order() if ZOO[m].group == GROUP]:
    if not conf[(conf.task == TASK) & (conf.scenario == SCENARIO) & (conf.model == m)].empty:
        display(viz.confusion(conf, TASK, SCENARIO, m))

## 10 · Robustness to sensor rotation
The score when the test data is rolled 15° and 30° about the body axis. This simulates the RESpeck being worn slightly differently.

In [ ]:
for task in [t for t in ["har", "social"] if t in set(runs.task)]:
    for v in [v for v in ("A", "D") if v in set(runs[runs.task == task].scenario)]:
        display(viz.rotation(runs, task, v))

## 11 · Accuracy vs cost on the phone
TFLite size and batch-1 latency, measured on the machine that ran the pipeline. The app needs one inference every 2 s, so anything under ~10 ms is negligible.

In [ ]:
for task in [t for t in ["har", "social"] if t in set(runs.task)]:
    ref = "D" if "D" in set(runs[runs.task == task].scenario) else "A"
    display(viz.deploy_tradeoff(runs, dep, task, ref))
if not dep.empty:
    display(dep.drop(columns=["scenario", "fold", "seed"], errors="ignore"))

## 12 · Training time

In [ ]:
for task in [t for t in ["har", "social"] if t in set(runs.task)]:
    display(viz.train_time(runs, task))

## 13 · Explore the raw numbers
`runs` has one row per trained model × test set. Filter and sort it any way you like; the example shows the 15 best social runs in scenario D.

In [ ]:
show = ["task", "scenario", "model", "owner", "fold", "seed", "SCORE", "STATIC_MEAN", "DYNAMIC_MEAN", "BALANCED", "MIN_CLASS",
        "window_acc", "macro_f1", "recording_vote_acc", "rot30", "params", "train_seconds"]
(runs[(runs.task == "social") & (runs.scenario == "D")]
 .sort_values("SCORE", ascending=False)[[c for c in show if c in runs.columns]].head(15).round(4))

## 14 · Previous experiments (2 Nov, before this pipeline)
These come from the earlier scripts (`results/`): real-data CV, the B9 runs, and her shipped `.tflite` files scored on the real data.

They used the older numpy augmentation and the CD scenario design, so compare them with sections 2–5 as context, not as a like-for-like row.

In [ ]:
prev, prev_pc = T.legacy(ROOT / "results")
if prev.empty:
    print("no previous results in results/")
else:
    for task in ["har", "social"]:
        display(viz.legacy_bars(prev, task))
        display(prev[prev.task == task].groupby(["scenario", "model"])[["SCORE", "STATIC_MEAN", "DYNAMIC_MEAN", "BALANCED", "MIN_CLASS"]]
                .mean().mul(100).round(1).dropna(axis=1, how="all"))